# 选修E1 · Day 1：Agent理论基础 · 上机练习（v5.0）

> **真实库**：LangChain + LangGraph + pydantic
> **核心范式**：ReAct（推理+行动）| Plan-Execute（规划+执行）
> **营销映射**：营销Agent的BDI认知结构 + 工具调用决策循环

本笔记本包含 **6个TODO填空**，完成后你将：
1. 用pydantic定义BDI Agent状态Schema
2. 用@tool装饰器定义营销工具
3. 用create_react_agent构建ReAct Agent
4. 运行Agent并分析Thought-Action-Observation轨迹
5. 用MemorySaver实现多轮对话记忆
6. 用StateGraph实现Plan-Execute模式

> 📦 真实库说明见 `data/README.md`
> 📖 理论讲义见 `notes.md`


In [ ]:
# === 导入真实库 ===
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, END
from pydantic import BaseModel, Field
from typing import Optional, TypedDict
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# === 真实营销数据（基于护肤品电商场景）===
PRODUCT_DB = {
    "透肌精华": "透肌焕亮精华液，299元，主打美白焕亮，含烟酰胺3%+维C衍生物，目标用户25-35岁都市白领。",
    "玻尿酸面霜": "玻尿酸保湿面霜，159元，主打深层补水，含双重玻尿酸，目标用户18-30岁女性。",
}
COMPETITOR_DB = {
    "雅诗兰黛": "雅诗兰黛小棕瓶精华，760元/30ml，市场占有率18%，优势：品牌力强、渠道完善；劣势：价格高、年轻化不足。",
    "兰蔻": "兰蔻小黑瓶精华，780元/30ml，市场占有率15%，优势：科技感强、专柜体验；劣势：下沉市场覆盖弱。",
}

# === 离线模拟LLM（无需API Key，预编排ReAct工具调用序列）===
class StubChatModel(BaseChatModel):
    """离线模拟LLM，预编排工具调用序列，保证无API Key可运行。
    替换为ChatOpenAI/ChatAnthropic即可使用真实LLM。"""
    responses: list = []
    call_index: int = 0

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        idx = self.call_index
        self.call_index += 1
        if idx < len(self.responses):
            resp = self.responses[idx]
        else:
            resp = AIMessage(content="任务完成。")
        return ChatResult(generations=[ChatGeneration(message=resp)])

    @property
    def _llm_type(self):
        return "stub"

    def bind_tools(self, tools, **kwargs):
        return self

print("真实库导入成功")
print(f"  产品库: {list(PRODUCT_DB.keys())}")
print(f"  竞品库: {list(COMPETITOR_DB.keys())}")
print(f"  StubChatModel: 离线模式（无API Key可运行）")


---
## TODO1：用pydantic定义BDI状态Schema

BDI（Belief-Desire-Intention）是经典Agent理论模型：
- **Belief（信念）**：Agent对世界的认知（产品信息、竞品信息、市场背景）
- **Desire（愿望）**：Agent想达成的目标（用户任务）
- **Intention（意图）**：Agent承诺执行的计划（步骤列表+当前步骤）

用pydantic的BaseModel将BDI形式化为Agent状态Schema，提供类型安全和自动校验。

> 参考教材 § Day 1 二、BDI架构

In [ ]:
# TODO1: 用pydantic定义BDI状态Schema

class Belief(BaseModel):
    """Agent对世界状态的认知（信念库）"""
    product_info: str = Field(default="", description="产品信息")
    competitor_info: str = Field(default="", description="竞品信息")
    market_context: str = Field(default="护肤品市场快速增长", description="市场背景")

class Desire(BaseModel):
    """Agent想要达成的目标（愿望集）"""
    task: str = Field(description="用户给定的营销任务")
    success_criteria: str = Field(default="策略文件已生成", description="成功标准")

class Intention(BaseModel):
    """Agent承诺执行的行动计划（意图队列）"""
    steps: list[str] = Field(default_factory=list, description="执行步骤列表")
    current_step: int = Field(default=0, description="当前步骤索引")

class BDIState(BaseModel):
    """BDI Agent完整状态"""
    belief: Belief = Field(default_factory=Belief)
    desire: Optional[Desire] = None
    intention: Optional[Intention] = None

# 验证BDI Schema
state = BDIState(
    desire=Desire(task="为透肌精华制定营销策略"),
    intention=Intention(steps=["搜索产品信息", "分析竞品", "撰写策略", "写入文件"])
)
print("BDI Schema定义成功")
print(f"  Belief: {state.belief.model_dump()}")
print(f"  Desire: {state.desire.model_dump()}")
print(f"  Intention: {state.intention.model_dump()}")


---
## TODO2：用@tool装饰器定义营销工具

工具是Agent的"手"。在LangChain中，用`@tool`装饰器定义工具。
工具的**名称、docstring、参数类型**就是LLM看到的"接口契约"。

需要定义三个营销工具：
1. `search_product_info(product_name)` - 搜索产品信息（使用PRODUCT_DB）
2. `analyze_competitor(competitor_name)` - 分析竞品策略（使用COMPETITOR_DB）
3. `write_strategy(filename, content)` - 将策略写入文件

> 参考教材 § Day 1 四、工具使用

In [ ]:
# TODO2: 用@tool装饰器定义三个营销工具

@tool
def search_product_info(product_name: str) -> str:
    """搜索产品信息。参数: product_name - 产品名称。返回产品详情（价格、功效、目标用户）。"""
    return PRODUCT_DB.get(product_name, f"未找到'{product_name}'的产品信息。可用产品: {list(PRODUCT_DB.keys())}")

@tool
def analyze_competitor(competitor_name: str) -> str:
    """分析竞争对手。参数: competitor_name - 竞品名称。返回竞品分析（价格、市占率、优劣势）。"""
    return COMPETITOR_DB.get(competitor_name, f"未找到'{competitor_name}'的竞品信息。可分析竞品: {list(COMPETITOR_DB.keys())}")

@tool
def write_strategy(filename: str, content: str) -> str:
    """将营销策略写入文件。参数: filename - 文件名; content - 策略内容。返回写入确认。"""
    return f"策略已写入 {filename}，共 {len(content)} 字。"

# 验证工具
tools = [search_product_info, analyze_competitor, write_strategy]
print(f"定义了 {len(tools)} 个营销工具:")
for t in tools:
    print(f"  - {t.name}: {t.description[:60]}...")

# 直接测试工具（不经过Agent）
result = search_product_info.invoke({"product_name": "透肌精华"})
print(f"\n工具测试: search_product_info('透肌精华') -> {result}")


---
## TODO3：用create_react_agent构建ReAct Agent

ReAct（Reasoning + Acting）的核心循环：
```
Thought -> Action -> Observation -> Thought -> ... -> FINISH
```

用LangGraph的`create_react_agent`构建ReAct Agent：
- model: 使用StubChatModel（预编排工具调用序列）
- tools: 使用TODO2定义的三个工具
- prompt: 系统提示，定义Agent角色

> 参考教材 § Day 1 三、ReAct范式

In [ ]:
# TODO3: 用create_react_agent构建ReAct Agent

# 预编排ReAct轨迹：模拟LLM的Thought-Action决策序列
react_trajectory = [
    AIMessage(content="", tool_calls=[{"name": "search_product_info", "args": {"product_name": "透肌精华"}, "id": "c1"}]),
    AIMessage(content="", tool_calls=[{"name": "analyze_competitor", "args": {"competitor_name": "雅诗兰黛"}, "id": "c2"}]),
    AIMessage(content="", tool_calls=[{"name": "write_strategy", "args": {"filename": "strategy.txt", "content": "差异化策略：主打性价比+年轻化定位。透肌精华299元vs雅诗兰黛760元，价格优势明显。目标用户25-35岁都市白领，注重性价比和成分透明。"}, "id": "c3"}]),
    AIMessage(content="营销策略已制定完成。基于产品分析（299元，美白焕亮，含烟酰胺3%）和竞品分析（760元，品牌力强但价格高），制定差异化策略：主打性价比+年轻化定位，已写入strategy.txt。"),
]

stub_model = StubChatModel(responses=react_trajectory)

# 构建ReAct Agent
agent = create_react_agent(
    stub_model,
    tools,
    prompt="你是一个营销策略Agent。根据用户需求，依次调用工具：搜索产品信息、分析竞品、撰写并写入策略。"
)

print("ReAct Agent构建成功")
print(f"  模型: {stub_model._llm_type} (离线StubLLM)")
print(f"  工具: {[t.name for t in tools]}")
print(f"  预编排轨迹: {len(react_trajectory)} 步")


---
## TODO4：运行Agent并分析ReAct轨迹

运行Agent处理营销任务，观察Thought-Action-Observation循环：
- 统计工具调用次数
- 统计模型调用次数
- 分析Agent的工具选择顺序是否符合天道推演的因果预期

> 天道推演视角：每个Action改变Belief，影响下一次Thought，形成因果链

In [ ]:
# TODO4: 运行Agent处理营销任务，观察ReAct轨迹

task = "为透肌精华制定营销策略，竞品分析雅诗兰黛，并写入策略文件"
result = agent.invoke({"messages": [("user", task)]})

print("=" * 60)
print("ReAct Agent执行轨迹")
print("=" * 60)

tool_calls_count = 0
thought_count = 0
for i, msg in enumerate(result["messages"]):
    msg_type = type(msg).__name__
    if msg_type == "HumanMessage":
        print(f"\n[Step {i}] Human: {msg.content}")
    elif msg_type == "AIMessage":
        if msg.tool_calls:
            for tc in msg.tool_calls:
                tool_calls_count += 1
                print(f"\n[Step {i}] Thought -> Action: {tc['name']}({tc['args']})")
        if msg.content:
            thought_count += 1
            print(f"\n[Step {i}] AI: {msg.content}")
    elif msg_type == "ToolMessage":
        print(f"\n[Step {i}] Observation: {msg.content}")

print("\n" + "=" * 60)
print("轨迹分析:")
print(f"  总消息数: {len(result['messages'])}")
print(f"  工具调用次数: {tool_calls_count}")
print(f"  模型调用次数: {stub_model.call_index}")
print(f"  ReAct循环: Thought({thought_count}) -> Action({tool_calls_count}) -> Observation({tool_calls_count})")


---
## TODO5：用MemorySaver实现多轮对话记忆

MemorySaver是LangGraph的checkpointer，按`thread_id`隔离不同会话。
同一thread_id的多轮对话共享上下文历史。

实现：
1. 构建带MemorySaver的Agent
2. 第一轮对话：查询产品信息
3. 第二轮对话：Agent应记住第一轮的内容

> 参考教材 § Day 1 关键回顾4：Agent记忆

In [ ]:
# TODO5: 用MemorySaver添加短期记忆，实现多轮对话

# 重新构建带记忆的Agent
stub_model_2 = StubChatModel(responses=[
    AIMessage(content="", tool_calls=[{"name": "search_product_info", "args": {"product_name": "玻尿酸面霜"}, "id": "m1"}]),
    AIMessage(content="玻尿酸保湿面霜信息已检索：159元，深层补水，目标用户18-30岁女性。"),
    AIMessage(content="是的，我记得您刚才查询的是玻尿酸保湿面霜，159元，主打深层补水。"),
])

memory = MemorySaver()
agent_with_memory = create_react_agent(
    stub_model_2,
    tools,
    prompt="你是一个营销策略Agent，有短期记忆，能记住当前对话上下文。",
    checkpointer=memory
)

# 第一轮对话
config = {"configurable": {"thread_id": "session-1"}}
print("=" * 60)
print("多轮对话测试（MemorySaver）")
print("=" * 60)

r1 = agent_with_memory.invoke(
    {"messages": [("user", "帮我查一下玻尿酸面霜的信息")]},
    config=config
)
print(f"\n[第1轮] 用户: 帮我查一下玻尿酸面霜的信息")
last_ai_1 = [m for m in r1["messages"] if type(m).__name__ == "AIMessage" and m.content][-1]
print(f"[第1轮] Agent: {last_ai_1.content}")

# 第二轮对话（Agent应记住第一轮的内容）
r2 = agent_with_memory.invoke(
    {"messages": [("user", "你刚才查的是什么产品？")]},
    config=config
)
print(f"\n[第2轮] 用户: 你刚才查的是什么产品？")
last_ai_2 = [m for m in r2["messages"] if type(m).__name__ == "AIMessage" and m.content][-1]
print(f"[第2轮] Agent: {last_ai_2.content}")

print(f"\nMemorySaver验证: 第2轮对话共{len(r2['messages'])}条消息（含第1轮历史）")
print(f"  thread_id='session-1' 隔离了会话上下文")


---
## TODO6：用StateGraph实现Plan-Execute模式

Plan-Execute与ReAct的核心区别：
- **ReAct**：边推理边执行，每步可根据观测调整下一步
- **Plan-Execute**：先一次性规划所有步骤，再顺序执行

用LangGraph的StateGraph实现：
1. `plan_node`：生成完整计划
2. `execute_node`：顺序执行每个步骤
3. 条件边：判断是否还有未执行步骤

> 参考教材 § Day 1 三、ReAct的局限性与改进

In [ ]:
# TODO6: 用StateGraph实现Plan-Execute模式

class PlanExecuteState(TypedDict):
    task: str
    plan: list[str]
    current_step: int
    results: list[str]
    final_answer: str

def plan_node(state):
    """规划节点：一次性生成完整计划"""
    task = state["task"]
    plan = [
        f"搜索产品信息: {task}",
        f"分析竞品策略: {task}",
        "撰写差异化策略",
        "写入策略文件",
    ]
    return {"plan": plan, "current_step": 0, "results": []}

def execute_node(state):
    """执行节点：执行当前步骤"""
    step_idx = state["current_step"]
    plan = state["plan"]
    if step_idx >= len(plan):
        return {"final_answer": "所有步骤已完成"}
    step_desc = plan[step_idx]
    if "搜索产品" in step_desc:
        result = PRODUCT_DB.get("透肌精华", "未找到")
    elif "分析竞品" in step_desc:
        result = COMPETITOR_DB.get("雅诗兰黛", "未找到")
    elif "撰写" in step_desc:
        result = "差异化策略：主打性价比+年轻化定位"
    elif "写入" in step_desc:
        result = "策略已写入 strategy.txt"
    else:
        result = f"执行: {step_desc}"
    return {
        "results": state["results"] + [result],
        "current_step": step_idx + 1,
    }

def should_continue(state):
    if state["current_step"] < len(state["plan"]):
        return "continue"
    return "end"

# 构建Plan-Execute图
graph = StateGraph(PlanExecuteState)
graph.add_node("plan", plan_node)
graph.add_node("execute", execute_node)
graph.set_entry_point("plan")
graph.add_edge("plan", "execute")
graph.add_conditional_edges("execute", should_continue, {
    "continue": "execute",
    "end": END,
})
plan_execute_agent = graph.compile()

# 运行Plan-Execute Agent
print("=" * 60)
print("Plan-Execute Agent执行结果")
print("=" * 60)

pe_result = plan_execute_agent.invoke({"task": "透肌精华"})
print(f"\n计划:")
for i, step in enumerate(pe_result["plan"]):
    print(f"  {i+1}. {step}")
print(f"\n执行结果:")
for i, (step, result) in enumerate(zip(pe_result["plan"], pe_result["results"])):
    print(f"  Step {i+1}: {step}")
    print(f"          -> {result}")

print(f"\n{'=' * 60}")
print("ReAct vs Plan-Execute 对比:")
print(f"  ReAct:        边推理边执行，{stub_model.call_index}次模型调用，3次工具调用")
print(f"  Plan-Execute: 先规划{len(pe_result['plan'])}步，顺序执行{len(pe_result['results'])}步")
print(f"  ReAct优势:    灵活适应，每步可根据观测调整")
print(f"  Plan-Execute优势: 全局计划，成本可预测，适合结构化任务")
